# Comparando curvas de luz (com CME e Sem CMEs)

## Proposta do script
O presente script tem como proposta a análise e comparação de eventos de CMEs analisados no Sol. 
1. Primeiramente, encontramos os sinais de eventos da CME (subtraindo curva de luz com CME da curva de lu sem CME)
2. Após encontrar os respectivos sinais de cada evento, conseguimos agrupar os padrões de cada sinal através da análise FFT e PCA
3. Ao encontrarmos os sinais do Sol, podemos comparar esses sinais com os sinais de estrelas hospedeiras. No caso do presente script, analisamos sinais em UV da base de dados XMM-Newton da estrela HD189733A


### Imports

In [42]:
#imports
import numpy as np
from matplotlib import pyplot
from Star.Estrela import Estrela
from Planet.Eclipse import Eclipse
from Planet.Planeta import Planeta
import numpy as np

%matplotlib tk

# Dados estrela

In [43]:
raio_estrela_pixel = 373. # default (pixel)
intensidade_maxima = 240 # default
tamanho_matriz = 856 # default
raio_estrela = 0.805 # raio da estrela em relacao ao raio do sol
coeficiente_um = 0.377
coeficiente_dois = 0.024

# Dados planeta

In [44]:
periodo = 2.219 # em dias
angulo_inclinacao = 85.51  # em graus
ecc = 0 # excentricidade
anomalia = 0 # anomalia
raio_plan_Jup = 1.138 # em relação ao raio de jupiter
semi_eixo_UA = 0.031 # UA
mass_planeta = 1.138 #em relacao ao R de jupiter

# 💥 Modelando a Estrela com CME 
## Exemplo inicial utilizando 2011/06/05

Aqui adicionamos o path contendo arquivos `.fits` referentes à data do envento a ser analisado. Para esse exemplo inicial, utilizamos o que está salvo no projeto, com o path `2011-06-05` dentro das pastas `Sun/sdo_aia_download`


In [45]:
#cria estrela
estrela_ = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2011-06-05")
tamanho_matriz = estrela_.getTamanhoMatriz()

Nx = estrela_.getNx() #Nx e Ny necessarios para a plotagem do eclipse
Ny = estrela_.getNy()
dtor = np.pi/180.  

## Modelando o Planeta

In [46]:
planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_.getRaioSun(), mass_planeta)

print(planeta_.getRaioPlan())


0.11425268977798202


In [47]:
estrela_matriz = estrela_.getMatrizEstrela()
estrela_.Plotar(tamanho_matriz, estrela_matriz)

## Eclipse com CME


In [48]:
eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_, planeta_, 1)
estrela_.Plotar(tamanho_matriz, estrela_matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()
curvaLuz_1 = curvaLuz
#Plotagem da curva de luz 
pyplot.plot(tempoHoras, curvaLuz)
pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz)-0.001, 1.001])                       
pyplot.show()



Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 6.699317837648663


In [49]:
latsugerida = eclipse_.calculaLatMancha()

A latitude sugerida para que a mancha influencie na curva de luz da estrela é: -30.794883579725727


# ☀ Modelando Estrela Sem CME
## Importante
_Para melhores resultados, compare o trecho com CME e sem CME no mesmo dia, assim evitamos ruídos extras devido à diferença de regiões ativas nas datas comparadas._

In [50]:
#cria estrela
estrela_sem_cme = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2011-06-05-no-cme")
tamanho_matriz = estrela_sem_cme.getTamanhoMatriz()

Nx = estrela_sem_cme.getNx() #Nx e Ny necessarios para a plotagem do eclipse
Ny = estrela_sem_cme.getNy()
dtor = np.pi/180.  

In [51]:
planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_sem_cme.getRaioSun(), mass_planeta)

print(planeta_.getRaioPlan())

0.11425268977798202


In [52]:
estrela_matriz = estrela_sem_cme.getMatrizEstrela()
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

In [53]:
#eclipse
eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_sem_cme, planeta_, 2)
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()
curvaLuz_2 = curvaLuz


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 6.699317837648663


# 📈 Comparando resultados

### Plotagem da curva de luz 

In [54]:
pyplot.plot(tempoHoras, curvaLuz_1, color="red", label="Curva de Luz com CME")
pyplot.plot(tempoHoras, curvaLuz_2, color="black", label="Curva de Luz sem CME")

pyplot.xlabel("Tempo (horas)")
pyplot.ylabel("Intensidade Normalizada")
pyplot.legend()  # Mostra a legenda
pyplot.title("Comparação entre curvas de luz com e sem CME do dia 2011/06/05")
pyplot.grid(True)
pyplot.show()

pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz)-0.001, 1.001])                       
pyplot.show()

## Sinal (diferença entre o horário com CME vs sem CME)

In [55]:
ruido = np.array(curvaLuz_1) - np.array(curvaLuz_2)

pyplot.plot(tempoHoras, ruido, label='Ruído da CME', alpha=0.5)
pyplot.xlabel('Tempo')
pyplot.xticks(rotation=65)
pyplot.ylabel('Fluxo Normalizado')
pyplot.legend()
pyplot.show()

# CME do dia 2017/04/24

In [56]:
# cria Estrela
estrela_3 = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2017-04-24")
tamanho_matriz = estrela_3.getTamanhoMatriz()

Nx = estrela_3.getNx() 
Ny = estrela_3.getNy()
dtor = np.pi/180.  

# cria Planeta
planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_3.getRaioSun(), mass_planeta)

estrela_3matriz = estrela_3.getMatrizEstrela()
estrela_3.Plotar(tamanho_matriz, estrela_3matriz)

# Eclipse com CME 

eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_3, planeta_, 1)
estrela_3.Plotar(tamanho_matriz, estrela_3matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz_3 = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()
#Plotagem da curva de luz 
pyplot.plot(tempoHoras, curvaLuz_3)
pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz_3)-0.001, 1.001])                       
pyplot.show()


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 6.699317837648663


In [57]:
#cria estrela sem CME
estrela_sem_cme = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2017-04-24-no-cme")
tamanho_matriz = estrela_sem_cme.getTamanhoMatriz()

Nx = estrela_sem_cme.getNx() #Nx e Ny necessarios para a plotagem do eclipse
Ny = estrela_sem_cme.getNy()
dtor = np.pi/180.  

planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_sem_cme.getRaioSun(), mass_planeta)
print(planeta_.getRaioPlan())


estrela_matriz = estrela_sem_cme.getMatrizEstrela()
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

#eclipse
eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_sem_cme, planeta_, 2)
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz_4 = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

0.11425268977798202

Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 6.699317837648663


In [58]:
# comparando resultados 
pyplot.plot(tempoHoras, curvaLuz_3, color="red", label="Curva de Luz com CME")
pyplot.plot(tempoHoras, curvaLuz_4, color="black", label="Curva de Luz sem CME")

pyplot.xlabel("Tempo (horas)")
pyplot.ylabel("Intensidade Normalizada")
pyplot.legend()  # Mostra a legenda
pyplot.title("Comparação entre curvas de luz com e sem CME do dia 2017/04/24")
pyplot.grid(True)
pyplot.show()

pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz_3)-0.001, 1.001])                       
pyplot.show()

In [59]:
ruido_2 = np.array(curvaLuz_3) - np.array(curvaLuz_4)

pyplot.plot(tempoHoras, ruido, label='Ruído da CME', alpha=0.5)
pyplot.xlabel('Tempo')
pyplot.xticks(rotation=65)
pyplot.ylabel('Fluxo Normalizado')
pyplot.legend()
pyplot.show()

# CME do dia 2017/04/30 (com [Flare](https://www.spaceweatherlive.com/en/archive/2017/04/30/xray.html), não HALO)

In [60]:
# cria Estrela
estrela_4 = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2017-04-30")
tamanho_matriz = estrela_4.getTamanhoMatriz()

Nx = estrela_4.getNx() 
Ny = estrela_4.getNy()
dtor = np.pi/180.  

# cria Planeta
angulo_inclinacao = 91.51  # em graus
planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_4.getRaioSun(), mass_planeta)

estrela_4matriz = estrela_4.getMatrizEstrela()
estrela_4.Plotar(tamanho_matriz, estrela_4matriz)

# Eclipse com CME 

eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_4, planeta_, 1)
estrela_4.Plotar(tamanho_matriz, estrela_4matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz_4 = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()
#Plotagem da curva de luz 
pyplot.plot(tempoHoras, curvaLuz_4)
pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz_4)-0.001, 1.001])                       
pyplot.show()


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 7.689306242422793


In [61]:
#cria estrela sem CME
estrela_sem_cme = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2017-04-30-no-cme")
tamanho_matriz = estrela_sem_cme.getTamanhoMatriz()

Nx = estrela_sem_cme.getNx() #Nx e Ny necessarios para a plotagem do eclipse
Ny = estrela_sem_cme.getNy()
dtor = np.pi/180.  

planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_sem_cme.getRaioSun(), mass_planeta)
print(planeta_.getRaioPlan())


estrela_matriz = estrela_sem_cme.getMatrizEstrela()
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

#eclipse
eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_sem_cme, planeta_, 2)
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz_5 = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

0.11425268977798202

Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 7.689306242422793


In [62]:
# comparando resultados 
pyplot.plot(tempoHoras, curvaLuz_4, color="red", label="Curva de Luz com CME")
pyplot.plot(tempoHoras, curvaLuz_5, color="black", label="Curva de Luz sem CME")

pyplot.xlabel("Tempo (horas)")
pyplot.ylabel("Intensidade Normalizada")
pyplot.legend()  # Mostra a legenda
pyplot.title("Comparação entre curvas de luz com e sem CME do dia 2017/04/30")
pyplot.grid(True)
pyplot.show()

pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz_4)-0.001, 1.001])                       
pyplot.show()

In [63]:
ruido_3 = np.array(curvaLuz_4) - np.array(curvaLuz_5)

pyplot.plot(tempoHoras, ruido, label='Ruído da CME', alpha=0.5)
pyplot.xlabel('Tempo')
pyplot.xticks(rotation=65)
pyplot.ylabel('Fluxo Normalizado')
pyplot.legend()
pyplot.show()

# CME do dia 2022/10/01 (com [Flare](https://www.spaceweatherlive.com/en/archive/2022/10/01/xray.html), HALO)

In [64]:
# cria Estrela
estrela_5 = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2022-10-01")
tamanho_matriz = estrela_5.getTamanhoMatriz()

Nx = estrela_5.getNx() 
Ny = estrela_5.getNy()
dtor = np.pi/180.  

# cria Planeta
angulo_inclinacao = 90  # em graus
planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_5.getRaioSun(), mass_planeta)

estrela_5matriz = estrela_5.getMatrizEstrela()
estrela_5.Plotar(tamanho_matriz, estrela_5matriz)

# Eclipse com CME 

eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_5, planeta_, 1)
estrela_5.Plotar(tamanho_matriz, estrela_5matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz_6 = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()
#Plotagem da curva de luz 
pyplot.plot(tempoHoras, curvaLuz_6)
pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz_6)-0.001, 1.001])                       
pyplot.show()


Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 7.807012980358783


In [65]:
#cria estrela sem CME
estrela_sem_cme = Estrela(raio_estrela_pixel, raio_estrela, intensidade_maxima, coeficiente_um, coeficiente_dois, tamanho_matriz, useFits = True, fits_path="2022-10-01-no-cme")
tamanho_matriz = estrela_sem_cme.getTamanhoMatriz()

Nx = estrela_sem_cme.getNx() #Nx e Ny necessarios para a plotagem do eclipse
Ny = estrela_sem_cme.getNy()
dtor = np.pi/180.  

planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_sem_cme.getRaioSun(), mass_planeta)
print(planeta_.getRaioPlan())


estrela_matriz = estrela_sem_cme.getMatrizEstrela()
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

#eclipse
eclipse_ = Eclipse(Nx, Ny, raio_estrela_pixel, estrela_sem_cme, planeta_, 2)
estrela_sem_cme.Plotar(tamanho_matriz, estrela_matriz)

tempoHoras = 1
eclipse_.geraTempoHoras(tempoHoras)
eclipse_.criarEclipse(anim=True)

print ("Tempo Total (Trânsito):", eclipse_.getTempoTransito()) 
tempoTransito = eclipse_.getTempoTransito()
curvaLuz_7 = eclipse_.getCurvaLuz()
tempoHoras = eclipse_.getTempoHoras()

0.11425268977798202

Aguarde um momento, a animacao do trânsito está sendo gerada...

Tempo Total (Trânsito): 7.807012980358783


In [66]:
# comparando resultados 
pyplot.plot(tempoHoras, curvaLuz_6, color="red", label="Curva de Luz com CME")
pyplot.plot(tempoHoras, curvaLuz_7, color="black", label="Curva de Luz sem CME")

pyplot.xlabel("Tempo (horas)")
pyplot.ylabel("Intensidade Normalizada")
pyplot.legend() 
pyplot.title("Comparação entre curvas de luz com e sem CME do dia 2022-10-01")
pyplot.grid(True)
pyplot.show()

pyplot.axis([-tempoTransito/2, tempoTransito/2, min(curvaLuz_6)-0.001, 1.001])                       
pyplot.show()

In [67]:
ruido_4 = np.array(curvaLuz_6) - np.array(curvaLuz_7)

pyplot.plot(tempoHoras, ruido_4, label='Ruído da CME', alpha=0.5)
pyplot.xlabel('Tempo')
pyplot.xticks(rotation=65)
pyplot.ylabel('Fluxo Normalizado')
pyplot.legend()
pyplot.show()

# CME do dia 2010/06/16

# Comparando Sinais
_Agora que temos todos os sinais de CMEs, podemos compará-los para tentar correlacioná-los de alguma forma_

In [68]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.signal import correlate

In [69]:
ruidos = [ruido, ruido_2, ruido_3, ruido_4]


# -------------------------------
# 1) Correlação cruzada
# -------------------------------
for i in range(len(ruidos)):
    for j in range(i+1, len(ruidos)):
        corr = correlate(ruidos[i], ruidos[j], mode="full")
        max_corr = np.max(np.abs(corr)) / len(ruidos[i])
        print(f"Correlação máxima entre ruido_{i+1} e ruido_{j+1}: {max_corr:.3f}")

# -------------------------------
# 2) PCA para padrões globais
# -------------------------------
X = np.vstack(ruidos)  # cada ruído é uma linha
X_std = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

print("Variância explicada por componente:", pca.explained_variance_ratio_)

# -------------------------------
# 3) FFT para espectro de frequência
# -------------------------------
fft_results = []
freqs = np.fft.fftfreq(len(ruidos[0]))  # eixo de frequência normalizado

for r in ruidos:
    fft_vals = np.abs(np.fft.fft(r))
    fft_results.append(fft_vals)

# -------------------------------
# 4) Visualização
# -------------------------------
plt.figure(figsize=(15,10))

# Plot 1: Ruídos no tempo
plt.subplot(2,2,1)
for i, r in enumerate(ruidos, start=1):
    plt.plot(r, label=f"Ruído {i}")
plt.title("Sinais de Ruído (Domínio do Tempo)")
plt.legend()

# Plot 2: PCA
plt.subplot(2,2,2)
plt.scatter(X_pca[:,0], X_pca[:,1], c=['b','orange','g','r'], s=100, label=f"Ruído {i+1}")
# for i in range(len(ruidos)):
#     plt.text(X_pca[i,0]-0.5, X_pca[i,1]-0.5, f"Ruído {i+1}")
plt.title("Projeção PCA (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")

# Plot 3: Espectro FFT
plt.subplot(2,1,2)
for i, fft_vals in enumerate(fft_results, start=1):
    plt.plot(freqs[:len(freqs)//2], fft_vals[:len(freqs)//2], label=f"Ruído {i}")
plt.title("Espectro de Frequência (FFT)")
plt.xlabel("Frequência normalizada")
plt.ylabel("Magnitude")
plt.legend()

plt.tight_layout()
plt.show()


Correlação máxima entre ruido_1 e ruido_2: 0.000
Correlação máxima entre ruido_1 e ruido_3: 0.000
Correlação máxima entre ruido_1 e ruido_4: 0.000
Correlação máxima entre ruido_2 e ruido_3: 0.000
Correlação máxima entre ruido_2 e ruido_4: 0.000
Correlação máxima entre ruido_3 e ruido_4: 0.000
Variância explicada por componente: [0.49916593 0.34852417]


# Análise CMEs utilizando dados de estrelas em Ultra Violeta
_A mesma análise de sinais pode ser feita em sinais de Ultra Violeta de outras estrelas. No caso abaixo, estamos utilizando dados coletados do XMM Newton em ultra-violeta do trânsito do planetal HD189733b_

## Média de eventos

In [29]:
from astropy.time import Time
from astropy.time import TimeDelta
import numpy as np
from Star.Estrela import Estrela, GeometriaCME
from Planet.Eclipse import Eclipse
from Planet.Planeta import Planeta
import matplotlib.pyplot as plt
import os
import glob
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt

### Padrão do arquivo .fitz

In [ ]:
padrao_arquivo = '*OM*TIMESR*.FTZ'

### Configuração do ambiente: funções necessárias

In [30]:
def calcular_transito(data_alvo_str, T0=2454279.436593, P=2.21857520, duracao_horas=1.8):
    """
    Calcula os tempos de início, meio и fim de um trânsito de HD 189733b
    próximo a uma data alvo, usando uma efeméride de alta precisão.

    Args:
        data_alvo_str (str): A data de interesse no formato 'AAAA-MM-DD HH:MM:SS'.
        T0 (float): A época de referência em JD.
        P (float): O período orbital em dias.
        duracao_horas (float): A duração total do trânsito em horas.

    Returns:
        dict: Um dicionário contendo os tempos do trânsito em formato JD e UTC.
    """
    # Converter a data de interesse para o objeto Time do Astropy
    t_alvo = Time(data_alvo_str, format='iso', scale='utc')
    
    # Calcular o número da órbita
    n_orbita = np.round((t_alvo.jd - T0) / P)
    
    # Calcular o tempo do meio do trânsito em JD
    jd_meio_transito = T0 + (n_orbita * P)
    
    # Converter o resultado JD para um objeto Time
    t_meio_transito = Time(jd_meio_transito, format='jd', scale='tdb')
    
    # --- CORREÇÃO APLICADA AQUI ---
    # Calcular início e fim do trânsito
    # Converte a duração de horas para segundos (1 hora = 3600s) e usa o formato 'sec'
    metade_duracao_segundos = (duracao_horas / 2.0) * 3600.0
    metade_duracao = TimeDelta(metade_duracao_segundos, format='sec')
    
    t_inicio_transito = t_meio_transito - metade_duracao
    t_fim_transito = t_meio_transito + metade_duracao
    
    # Preparar o dicionário de resultados
    resultados = {
        'orbita_n': int(n_orbita),
        'meio_transito_utc': t_meio_transito.utc.iso,
        'inicio_transito_utc': t_inicio_transito.utc.iso,
        'fim_transito_utc': t_fim_transito.utc.iso,
        'meio_transito_jd': jd_meio_transito
    }
    
    return resultados

# --- Exemplo de Uso ---
data_teste = '2014-11-15 12:00:00'
T0 = 2454279.436593
P = 2.21857520
duracao_horas = 1.8 # (e.g., Torres et al. 2008)
transito_previsto = calcular_transito(data_teste, T0, P, duracao_horas)
print(transito_previsto)

{'orbita_n': 1216, 'meio_transito_utc': '2014-11-15 17:21:29.545', 'inicio_transito_utc': '2014-11-15 16:27:29.545', 'fim_transito_utc': '2014-11-15 18:15:29.545', 'meio_transito_jd': 2456977.2240362}


In [31]:
def modelo_de_transito_eclipse(tempo_horas_alvo, parametros):
    """
    Esta função usa o código ECLIPSE para gerar um modelo de trânsito.
    """
    
    # --------------------- Estrela ---------------------
    raio = 373.
    intensidadeMaxima = 240
    tamanhoMatriz = 856
    u1 = parametros['u1']
    u2 = parametros['u2']

    # --------------------- Planeta ---------------------
    raio_plan_Jup = parametros['raio_plan_Jup']
    semi_eixo_UA = parametros['semi_eixo_UA']
    angulo_inclinacao = parametros['angulo_inclinacao']
    periodo = parametros['periodo']
    mass_planeta = 1.138
    ecc = 0
    anomalia = 0
    rsun = parametros['rsun']
    
    estrela_ = Estrela(raio, rsun, intensidadeMaxima, u1, u2, tamanhoMatriz)
    Nx = estrela_.getNx()
    Ny = estrela_.getNy()
    raioEstrelaPixel = estrela_.getRaioStar()

    planeta_ = Planeta(semi_eixo_UA, raio_plan_Jup, periodo, angulo_inclinacao, ecc, anomalia, estrela_.getRaioSun(), mass_planeta)

    eclipse = Eclipse(Nx, Ny, raioEstrelaPixel, estrela_, planeta_,1)
    
    # Define a duração total da simulação para garantir que o trânsito completo seja gerado
    # Por exemplo, 5 horas é suficiente para um trânsito de 1.8h
    eclipse.setTempoHoras(5.) 

    eclipse.criarEclipse(anim=False, plot=False) # plot=False para não gerar gráficos intermediários
    
    lc0 = np.array(eclipse.getCurvaLuz()) 
    ts0 = np.array(eclipse.getTempoHoras())

    # --- ETAPA DE ALINHAMENTO E INTERPOLAÇÃO ---
    # 1. Normalizar o modelo Eclipse (caso ele não esteja entre 0 e 1)
    lc0_normalizado = lc0 / np.max(lc0)
    
    # 2. Encontrar o tempo do centro do trânsito no modelo
    tempo_centro_modelo_simulado = ts0[np.argmin(lc0_normalizado)]

    # 3. Deslocar o eixo de tempo do modelo para que o centro do trânsito
    #    se alinhe com o centro que medimos nos dados observados (t0)
    ts0_alinhado = ts0 - tempo_centro_modelo_simulado + parametros['t0']

    # 4. Interpolar o modelo para os pontos de tempo do nosso gráfico final
    #    Isso cria a linha suave e contínua
    fluxo_final_modelo = np.interp(tempo_horas_alvo, ts0_alinhado, lc0_normalizado)
    
    return fluxo_final_modelo

def busca_fluxo_normalizado(lista_arquivos): 
    todos_tempos, todos_fluxos, todos_erros = [], [], []
    for arquivo in lista_arquivos:
        with fits.open(arquivo) as hdul:
            dados = hdul[1].data
            todos_tempos.extend(dados['TIME'])
            todos_fluxos.extend(dados['RATE'])
            todos_erros.extend(dados['ERROR'])
    todos_tempos, todos_fluxos, todos_erros = np.array(todos_tempos), np.array(todos_fluxos), np.array(todos_erros)
    indices_ordenados = np.argsort(todos_tempos)
    tempos_ordenados = todos_tempos[indices_ordenados]
    fluxos_ordenados = todos_fluxos[indices_ordenados]
    tempos_em_horas = (tempos_ordenados - tempos_ordenados[0]) / 3600.0
    fluxos_normalizados = fluxos_ordenados / np.mean(fluxos_ordenados[tempos_em_horas > 3.0])
    return tempos_em_horas, fluxos_normalizados


def bin_data(tempos_em_horas, fluxos_normalizados, bin = 0.2): 
    bin_size_horas = bin
    bins = np.arange(tempos_em_horas.min(), tempos_em_horas.max(), bin_size_horas)
    binned_time, binned_flux, binned_error = [], [], []

    for i in range(len(bins) - 1):
        mask = (tempos_em_horas >= bins[i]) & (tempos_em_horas < bins[i+1])
        if np.sum(mask) > 0:
            binned_time.append(np.mean(tempos_em_horas[mask]))
            binned_flux.append(np.mean(fluxos_normalizados[mask]))
            n_points = len(fluxos_normalizados[mask])
            binned_error.append(np.std(fluxos_normalizados[mask]) / np.sqrt(n_points))

    
    return binned_time, binned_flux, binned_error

def plot_lightcurve_and_model(binned_time, binned_flux, binned_error, tempo_modelo_plot, fluxo_do_seu_modelo, date_obs, id_obs): 
    plt.figure(figsize=(16, 8))
    plt.errorbar(binned_time, binned_flux, yerr=binned_error, fmt='o', markersize=8, color='purple', ecolor='purple', capsize=4, label=f'Dados Observados (Binados)')
    plt.plot(tempo_modelo_plot, fluxo_do_seu_modelo, color='red', linewidth=2.5, label='Modelo ECLIPSE')
    plt.axhline(1.0, color='black', linestyle='--', alpha=0.7, label='Linha de Base')
    plt.title('Modelo de Trânsito para HD 189733 vs Dados Observados: ' + date_obs + " ID:" + id_obs)
    plt.xlabel('Tempo (horas desde o início da primeira observação)')
    plt.ylabel('Fluxo Normalizado')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.ylim(0.95, 1.05)
    plt.tight_layout()
    plt.show()

In [34]:
def processar_curva_de_luz_para_empilhar(pasta_dados, t0_centro_transito, periodo_dias):
    """
    Função para carregar e processar uma curva de luz.
    Retorna os dados NÃO-BINADOS, mas já em fase orbital.
    """
    padrao_arquivo = '*OM*TIMESR*.FTZ'
    caminho_busca = os.path.join(pasta_dados, padrao_arquivo)
    lista_arquivos = glob.glob(caminho_busca)
    
    todos_tempos, todos_fluxos, todos_erros = [], [], []
    for arquivo in lista_arquivos:
        with fits.open(arquivo) as hdul:
            dados = hdul[1].data
            todos_tempos.extend(dados['TIME'])
            todos_fluxos.extend(dados['RATE'])
            todos_erros.extend(dados['ERROR'])
    
    todos_tempos, todos_fluxos, todos_erros = np.array(todos_tempos), np.array(todos_fluxos), np.array(todos_erros)
    indices_ordenados = np.argsort(todos_tempos)
    tempos_ordenados = todos_tempos[indices_ordenados]
    fluxos_ordenados = todos_fluxos[indices_ordenados]
    erros_ordenados = todos_erros[indices_ordenados]
    
    tempos_em_horas = (tempos_ordenados - tempos_ordenados[0]) / 3600.0
    
    mask_fora_transito = np.abs(tempos_em_horas - t0_centro_transito) > 1.5
    fluxo_fora_transito = np.mean(fluxos_ordenados[mask_fora_transito])
    fluxos_normalizados = fluxos_ordenados / fluxo_fora_transito
    erros_normalizados = erros_ordenados / fluxo_fora_transito
    
    periodo_em_horas = periodo_dias * 24.0
    fase_observada = (tempos_em_horas - t0_centro_transito) / periodo_em_horas
            
    return fase_observada, fluxos_normalizados, erros_normalizados

### Parâmetros do modelo

In [ ]:
# Modelo inicial para a estrela estudada HD189733A
parametros_modelo = {
    'rsun': 0.805,
    'u1': 0.0618,
    'u2': 0.0204,
    'raio_plan_Jup': 1.138,
    'semi_eixo_UA': 0.031,
    'angulo_inclinacao': 85.710,
    'periodo': 2.21857520,
    # Parâmetro medido dos dados para alinhamento
    't0': 1 # Centro do trânsito default
}

O período aqui adicionado é importante para o cálculo do centro do trânsito. Quanto mais preciso, melhor o posicionamento do modelo na curva de luz em UV observada

In [1]:
periodo_geral_dias = 2.21857520

### Configuração das observações baixadas
No caso, baixamos 10 observações em UV da base de dados XMM-newton. O Local Path está configurado com o meu pessoal, aqui você deve alterar para o seu path local pessoal com as suas observações. 

### Importante
t0: configura o tempo central do trânsito observado em cada curva de luz

In [35]:
# --- 1. CONFIGURAÇÃO ---
observacoes = [
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0506070201/pps/', 
        't0': 9.5161,
        'label': '2007-04-17'
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0690890201/pps/',
        't0': 7.4245,
        'label': '2012-06-07'
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0672390201/pps/', 
        't0': 7.8336,
        'label': '2011-05-01' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744981401/pps/', 
        't0': 5.2427,
        'label': '2014-11-13' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744981601/pps/', 
        't0': 6.8033,
        'label': '2015-04-17' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744981701/pps/', 
        't0': 4.5239,
        'label': '2015-04-19' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744980801/pps/', 
        't0': 4.695,
        'label': '2014-10-1' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744981201/pps/', 
        't0': 6.1414,
        'label': '2014-11-11' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744980601/pps/', 
        't0': 4.8147,
        'label': '2014-05-17' 
    },
    {
        'pasta': '/Users/beatrizduque/Downloads/Mestrado/HD189733/ligthcurves-om/0744980701/pps/', 
        't0': 7.4447,
        'label': '2014-11-15' 
    },
]


# --- 2. PROCESSAR E COMBINAR
todas_as_fases, todos_os_fluxos = [], []
for obs in observacoes:
    print(f" -> Processando {obs['label']}...")
    fase, fluxo, _ = processar_curva_de_luz_para_empilhar(obs['pasta'], obs['t0'], periodo_geral_dias)

    todas_as_fases.append(fase)
    todos_os_fluxos.append(fluxo)

print("Processamento concluído!")
fases_combinadas = np.concatenate(todas_as_fases)
fluxos_combinados = np.concatenate(todos_os_fluxos)

# --- 3. BINAR O CONJUNTO COMBINADO
bin_size_fase = 0.0025
bins = np.arange(fases_combinadas.min(), fases_combinadas.max(), bin_size_fase)
binned_fase, binned_flux, binned_error = [], [], []
for i in range(len(bins) - 1):
    mask = (fases_combinadas >= bins[i]) & (fases_combinadas < bins[i+1])
    if np.sum(mask) > 1:
        binned_fase.append(np.mean(fases_combinadas[mask]))
        binned_flux.append(np.mean(fluxos_combinados[mask]))
        n_points = len(fluxos_combinados[mask])
        binned_error.append(np.std(fluxos_combinados[mask]) / np.sqrt(n_points))

# --- 4. GERAR O MODELO TEÓRICO ---
parametros_modelo = {
    'rsun': 0.805, 
    'u1': 0.377, 
    'u2': 0.024, 
    'raio_plan_Jup': 1.138,
    'semi_eixo_UA': 0.031, 
    'angulo_inclinacao': 85.51, 
    'periodo': periodo_geral_dias,
    't0': 0.0 # O centro do nosso modelo é em t=0 horas, pois vamos plotar em fase
}

# Criar um eixo de fase suave para a linha do modelo
fase_modelo_plot = np.linspace(-0.025, 0.025, 1000)
# Converter o eixo de fase para um eixo de tempo em horas, centrado em zero
periodo_em_horas = periodo_geral_dias * 24.0
tempo_modelo_horas = fase_modelo_plot * periodo_em_horas
# Chamar sua função de modelo para gerar o fluxo
fluxo_do_seu_modelo = modelo_de_transito_eclipse(tempo_modelo_horas, parametros_modelo)


# --- 5. PLOTAGEM FINAL ---
plt.figure(figsize=(12, 8))

# Plota a curva de luz média binada
plt.errorbar(binned_fase, binned_flux, yerr=binned_error, fmt='o', color='purple', markersize=6, ecolor='purple', capsize=0, label=f'Curva de Luz Média ({len(observacoes)} Trânsitos)')

# Plota o seu modelo teórico por cima
plt.plot(fase_modelo_plot, fluxo_do_seu_modelo, color='red', linewidth=2.5, zorder=10, label='Modelo Teórico')

plt.title(f'Modelo Teórico vs. Curva de Luz Média de HD 189733b', fontsize=16)
plt.xlabel('Fase Orbital (0 = Meio do Trânsito)', fontsize=12)
plt.ylabel('Fluxo Normalizado', fontsize=12)
plt.xlim(-0.025, 0.025)
plt.ylim(0.965, 1.035)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

 -> Processando 2007-04-17...
 -> Processando 2012-06-07...
 -> Processando 2011-05-01...
 -> Processando 2014-11-13...
 -> Processando 2015-04-17...
 -> Processando 2015-04-19...
 -> Processando 2014-10-1...
 -> Processando 2014-11-11...
 -> Processando 2014-05-17...
 -> Processando 2014-11-15...
Processamento concluído!


### Busca por sinais de CMEs
Aqui comparamos cada observação individualmente com a média das 10 observações. Assim, ao entendermos a que mais se afasta da média, podemos encontrar algum tipo de sinal que represente algum evento. No caso do estudo, esse evento pode ser uma CME

In [36]:
indice_da_obs_alvo = 2 # interessante 2 

fase_media, fluxo_medio, erro_medio = binned_fase, binned_flux, binned_error

# --- 3. PROCESSAR A OBSERVAÇÃO ALVO INDIVIDUALMENTE ---
obs_alvo = observacoes[indice_da_obs_alvo]
fase_alvo, fluxo_alvo, erro_alvo = processar_curva_de_luz_para_empilhar(obs_alvo['pasta'], obs_alvo['t0'], periodo_geral_dias)

# Binning da observação alvo para ter menos pontos
binned_fase_alvo, binned_fluxo_alvo, binned_erro_alvo = [], [], []
bins_alvo = np.arange(fase_alvo.min(), fase_alvo.max(), bin_size_fase)
for i in range(len(bins_alvo) - 1):
    mask = (fase_alvo >= bins_alvo[i]) & (fase_alvo < bins_alvo[i+1])
    if np.sum(mask) > 0: # > 0 para não perder bins com poucos pontos
        binned_fase_alvo.append(np.mean(fase_alvo[mask]))
        binned_fluxo_alvo.append(np.mean(fluxo_alvo[mask]))
        n_points = len(fluxo_alvo[mask])
        binned_erro_alvo.append(np.std(fluxo_alvo[mask]) / np.sqrt(n_points))
binned_fase_alvo, binned_fluxo_alvo, binned_erro_alvo = np.array(binned_fase_alvo), np.array(binned_fluxo_alvo), np.array(binned_erro_alvo)


# --- 4. SUBTRAIR A MÉDIA DA OBSERVAÇÃO ALVO ---
# Interpolar a curva média para os pontos de fase da observação alvo
fluxo_medio_interp = np.interp(binned_fase_alvo, fase_media, fluxo_medio)
erro_medio_interp = np.interp(binned_fase_alvo, fase_media, erro_medio)

# Calcular os resíduos
residuos = binned_fluxo_alvo - fluxo_medio_interp
erro_residuos = np.sqrt(binned_erro_alvo**2 + erro_medio_interp**2)

# --- 5. ANÁLISE ESTATÍSTICA DOS RESÍDUOS ---
modelo = np.zeros_like(residuos)
dados_observados = residuos
incerteza = erro_residuos
qui_quadrado = np.sum(((dados_observados - modelo) / incerteza)**2)
graus_de_liberdade = len(dados_observados) - 1
qui_quadrado_reduzido = qui_quadrado / graus_de_liberdade

print("-" * 50)
print("ANÁLISE ESTATÍSTICA DOS RESÍDUOS OBS", indice_da_obs_alvo+1)
print(f"(Comparando a observação de {obs_alvo['label']} com a média)")
print("-" * 50)
print(f"Valor do Qui-quadrado Reduzido (χ²ν): {qui_quadrado_reduzido:.4f}")

if 0.7 <= qui_quadrado_reduzido <= 1.5:
    print("Interpretação: Excelente! Um valor próximo de 1.0 indica que a diferença")
    print("entre esta observação e a média é consistente com o ruído esperado.")
    print("==> Conclusão Numérica: A observação alvo é estatisticamente 'normal'.")
elif qui_quadrado_reduzido > 1.5:
    print("Interpretação: Atenção. Um valor significativamente maior que 1.0 sugere")
    print("que a diferença entre esta observação e a média é maior que o ruído esperado.")
    print("==> Conclusão Numérica: Esta observação pode conter um evento anômalo.")
else: 
    print("Interpretação: Interessante. Um valor muito menor que 1.0 pode indicar")
    print("que as barras de erro dos dados foram superestimadas.")
print("-" * 50)

# --- 6. PLOTAGEM FINAL ---
fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle(f'Análise de Resíduos: Observação de {obs_alvo["label"]} vs. Média', fontsize=16)

# Painel Superior: Comparação
axs[0].errorbar(binned_fase_alvo, binned_fluxo_alvo, yerr=binned_erro_alvo, fmt='o', color='purple', label=f"Obs Alvo ({obs_alvo['label']})")
axs[0].errorbar(fase_media, fluxo_medio, yerr=erro_medio, fmt='o', color='black', label=f"Obs Média de {len(observacoes)} Trânsitos")
axs[0].plot(fase_modelo_plot, fluxo_do_seu_modelo, color='red', linewidth=2.5, zorder=10, label='Modelo Teórico')
#axs[0].plot(fase_media, fluxo_medio, color='black', linewidth=2, label=f"Média de {len(observacoes)} Trânsitos")
axs[0].set_title('Observação Individual vs. Curva Média')
axs[0].set_ylabel('Fluxo Normalizado')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].legend()

# Painel Inferior: Resíduos
axs[1].errorbar(binned_fase_alvo, residuos, yerr=erro_residuos, fmt='o', color='crimson', markersize=5)
axs[1].axhline(0, color='black', linestyle='--')
# --- MUDANÇA: Adiciona o valor de chi2 ao título do gráfico de resíduos ---
axs[1].set_title(f'Resíduos (Obs Individual - Média)  |  $\\chi_\\nu^2 = {qui_quadrado_reduzido:.2f}$')
axs[1].set_xlabel('Fase Orbital (0 = Meio do Trânsito)')
axs[1].set_ylabel('Diferença')
axs[1].grid(True, linestyle='--', alpha=0.6)

plt.xlim(-0.025, 0.025)
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

--------------------------------------------------
ANÁLISE ESTATÍSTICA DOS RESÍDUOS OBS 3
(Comparando a observação de 2011-05-01 com a média)
--------------------------------------------------
Valor do Qui-quadrado Reduzido (χ²ν): 1.8721
Interpretação: Atenção. Um valor significativamente maior que 1.0 sugere
que a diferença entre esta observação e a média é maior que o ruído esperado.
==> Conclusão Numérica: Esta observação pode conter um evento anômalo.
--------------------------------------------------


### Filtro

In [106]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from scipy.signal import savgol_filter

In [107]:
binned_fluxo_medio_smooth = savgol_filter(fluxo_medio, window_length=11, polyorder=3)
binned_fluxo_alvo_smooth = savgol_filter(binned_fluxo_alvo, window_length=11, polyorder=3)

In [108]:
fig, axs = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
fig.suptitle(f'Análise de Resíduos: Observação de {obs_alvo["label"]} vs. Média', fontsize=16)

axs[0].errorbar(binned_fase_alvo, binned_fluxo_alvo_smooth, yerr=binned_erro_alvo, fmt='o', color='purple', label=f"Obs Alvo ({obs_alvo['label']})")
axs[0].errorbar(fase_media, binned_fluxo_medio_smooth, yerr=erro_medio, fmt='o', color='black', label=f"Obs Média de {len(observacoes)} Trânsitos")
axs[0].plot(fase_modelo_plot, fluxo_do_seu_modelo, color='red', linewidth=2.5, zorder=10, label='Seu Modelo Teórico')

axs[0].set_title('Observação Individual vs. Curva Média')
axs[0].set_ylabel('Fluxo Normalizado')
axs[0].grid(True, linestyle='--', alpha=0.6)
axs[0].legend()

axs[1].errorbar(binned_fase_alvo, residuos, yerr=erro_residuos, fmt='o', color='crimson', markersize=5)
axs[1].axhline(0, color='black', linestyle='--')

axs[1].set_title(f'Resíduos (Obs Individual - Média)  |  $\\chi_\\nu^2 = {qui_quadrado_reduzido:.2f}$')
axs[1].set_xlabel('Fase Orbital (0 = Meio do Trânsito)')
axs[1].set_ylabel('Diferença')
axs[1].grid(True, linestyle='--', alpha=0.6)

plt.xlim(-0.025, 0.025)
plt.tight_layout(rect=[0, 0.03, 1, 0.96])

# Comparação do Sinal da CME da estrela HD189733A com os sinais de CMEs do Sol

Agora que selecionamos a observação número 3 como a que possívelmente possui algum evento, podemos então comparar esse sinal com os sinais anteriormente analisados do Sol 

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

In [119]:
# =================================================================
# 1. ALINHAMENTO DO MODELO TEÓRICO COM OS RESÍDUOS
# =================================================================

# 1. Interpolar o Modelo Teórico para o Eixo X da Observação Alvo
# Os pontos do modelo ('fase_modelo_plot', 'fluxo_do_seu_modelo') são a base.
f_interp_modelo = interp1d(fase_modelo_plot, fluxo_do_seu_modelo, kind='linear', fill_value="extrapolate")

# O Modelo Teórico Alinhado (M_alinhado)
fluxo_modelo_alinhado = f_interp_modelo(binned_fase_alvo)

# 2. Calcular o NOVO RESÍDUO: (Obs. Alvo - Modelo Teórico Alinhado)
# Cálculo da diferença entre a Observação e o Modelo Teórico.
# Este é o array que realmente isola o sinal de dimming, ruído, e anomalias.
residuos_modelo = binned_fluxo_alvo_smooth - fluxo_modelo_alinhado

# =================================================================
# 2. DEFINIÇÃO DA VARIÁVEL PARA ANÁLISE (ruido_5)
# =================================================================

# O 'ruido_5' agora representa a diferença entre a observação e o modelo teórico perfeito.
# Esta é a melhor métrica para a sua análise PCA.
ruido_5_analise = residuos_modelo

# Exemplo de como você plotaria este novo resíduo (opcional):
# plt.figure()
# plt.errorbar(binned_fase_alvo, residuos_modelo, fmt='o', color='blue', label='Obs Alvo - Modelo Teórico')
# plt.axhline(0, color='black', linestyle='--')
# plt.title('Novo Resíduo: Sinal de Dimming Isolado')
# plt.show()

### Tratamento de daddos

In [121]:
# --- 1. DEFINIÇÃO DA VARIÁVEL DE COMPRIMENTO DE BASE ---
# 856 é o tamanho dos ruídos originais.
N_base = len(ruido) 

# --- 2. CONFIGURAÇÃO DE INPUT DO RUÍDO 5 ---
# Certifique-se de que estas variáveis estejam disponíveis no seu ambiente:
ruido_5_raw = residuos_modelo      # O array Y que você acabou de calcular
fase_ruido_5_raw = binned_fase_alvo # O eixo X correspondente

# --- 3. EXECUÇÃO DA INTERPOLAÇÃO PARA AJUSTAR O TAMANHO ---
# 3.1. Criar o novo eixo X (tempo/fase) com 856 pontos
fase_base_uniforme = np.linspace(fase_ruido_5_raw.min(), fase_ruido_5_raw.max(), N_base)

# 3.2. Criar a função de interpolação
f_interp = interp1d(fase_ruido_5_raw, ruido_5_raw, kind='linear', fill_value="extrapolate")

# 3.3. Aplicar a interpolação: O resultado terá N_base = 856 pontos.
ruido_5_ajustado = f_interp(fase_base_uniforme)

# --- 4. DEFINIÇÃO DA VARIÁVEL FINAL ---
ruido_5 = ruido_5_ajustado

### RECONSTRUÇÃO DA LISTA
_Adicionando o Sinal em UV da estrela ao array com sinais do Sol_

Origem dos dados
```
ruido: Sol
ruido_2: Sol
ruido_3: Sol
ruido_4: Sol
ruido_5: Estrela HD189733A
```

In [ ]:
# 1. Lista bruta dos dados de Sinais de CME, adicionando como ruido_5 o sinal da CME da estrela HD189733A
ruidos_brutos = [ruido, ruido_2, ruido_3, ruido_4, ruido_5]

num_ruidos = len(ruidos_brutos)
nomes_ruidos = [f"Ruído {i}" for i in range(1, num_ruidos + 1)]
N_base = len(ruidos_brutos[0]) # Comprimento base para FFT/Correlação

### Normalização e tratamento de dados

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [123]:
# 2. Normalização Essencial (Normaliza cada sinal individualmente por seu STD)
ruidos_normalizados = []
for r in ruidos_brutos:
    std_dev = np.std(r)
    if std_dev > 1e-10:
        # Normaliza (Divide pelo desvio padrão)
        r_norm = r / std_dev 
    else:
        r_norm = r
    ruidos_normalizados.append(r_norm)

# Usamos 'ruidos_normalizados' para PCA e 'ruidos_brutos' para FFT (para magnitude real)

# =================================================================
# 2. ANÁLISE ESTATÍSTICA E ESPECTRAL
# =================================================================

# -------------------------------
# 1) Correlação cruzada
# -------------------------------
print("\n--- Correlação Cruzada ---")
# Usaremos os ruídos BRUTOS aqui (a normalização pode mascarar a correlação de magnitude)
for i in range(num_ruidos):
    for j in range(i+1, num_ruidos):
        corr = correlate(ruidos_brutos[i], ruidos_brutos[j], mode="full")
        max_corr = np.max(np.abs(corr)) / N_base 
        print(f"Correlação máxima entre {nomes_ruidos[i]} e {nomes_ruidos[j]}: {max_corr:.3f}")

# -------------------------------
# 2) PCA para padrões globais
# -------------------------------
print("\n--- Análise PCA ---")
X = np.vstack(ruidos_normalizados) # PCA usa os dados normalizados por STD
X_std = StandardScaler().fit_transform(X) # Standard Scaler (média=0, variância=1)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_std)

var_explicada = pca.explained_variance_ratio_
print(f"Variância explicada por PC1: {var_explicada[0]*100:.1f}%")
print(f"Variância explicada por PC2: {var_explicada[1]*100:.1f}%")

# -------------------------------
# 3) FFT para espectro de frequência
# -------------------------------
fft_results = []
freqs = np.fft.fftfreq(N_base)

for r in ruidos_brutos: # FFT usa dados BRUTOS (para manter a magnitude real)
    fft_vals = np.abs(np.fft.fft(r))
    fft_results.append(fft_vals)

# =================================================================
# 3. VISUALIZAÇÃO
# =================================================================

plt.figure(figsize=(15, 12))
cores = ['b', 'orange', 'g', 'r', 'm'] 

# Plot 1: Ruídos no tempo (Usando os dados BRUTOS para ver a amplitude real)
plt.subplot(2, 2, 1)
for i, r in enumerate(ruidos_brutos):
    plt.plot(r, label=nomes_ruidos[i], color=cores[i])
plt.title("Sinais de Ruído (Domínio do Tempo)")
plt.xlabel("Índice do Ponto (ou Fase Interp.)")
plt.ylabel("Fluxo Residual")
plt.legend()

# Plot 2: PCA (Usando os dados NORMALIZADOS para análise de forma)
plt.subplot(2, 2, 2)
plt.scatter(X_pca[:,0], X_pca[:,1], c=cores, s=150)
for i, txt in enumerate(nomes_ruidos):
     # Adiciona rótulos ao lado dos pontos
     plt.annotate(txt, (X_pca[i, 0] + 0.1, X_pca[i, 1])) 
plt.title("Projeção PCA (2D)")
plt.xlabel(f"PC1 ({var_explicada[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({var_explicada[1]*100:.1f}%)")
plt.grid(True, linestyle='--', alpha=0.5)

# Plot 3: Espectro FFT (Usando dados BRUTOS)
plt.subplot(2, 1, 2)
for i, fft_vals in enumerate(fft_results):
    plt.plot(freqs[:N_base//2], fft_vals[:N_base//2], label=nomes_ruidos[i], color=cores[i])
plt.title("Espectro de Frequência (FFT)")
plt.xlabel("Frequência normalizada")
plt.ylabel("Magnitude")
plt.legend()

plt.tight_layout()
plt.show()


--- Correlação Cruzada ---
Correlação máxima entre Ruído 1 e Ruído 2: 0.000
Correlação máxima entre Ruído 1 e Ruído 3: 0.000
Correlação máxima entre Ruído 1 e Ruído 4: 0.000
Correlação máxima entre Ruído 1 e Ruído 5: 0.000
Correlação máxima entre Ruído 2 e Ruído 3: 0.000
Correlação máxima entre Ruído 2 e Ruído 4: 0.000
Correlação máxima entre Ruído 2 e Ruído 5: 0.000
Correlação máxima entre Ruído 3 e Ruído 4: 0.000
Correlação máxima entre Ruído 3 e Ruído 5: 0.000
Correlação máxima entre Ruído 4 e Ruído 5: 0.000

--- Análise PCA ---
Variância explicada por PC1: 82.0%
Variância explicada por PC2: 9.2%


invalid command name "exit"
    while executing
"exit"


A Análise de Componentes Principais (PCA) revela que a principal fonte de variância (PC1, >82%) é a diferença inerente de amplitude entre os canais de observação. O Ruído 5 (UV Próximo) exibe variações de magnitude significativamente maior do que os Ruídos 1-4 (UV Extremo). No entanto, o PCA ainda isola o Ruído 4 (e o Ruído 3 em menor grau) no PC2, confirmando que, mesmo dentro do cluster de baixa amplitude do UV Extremo, esses sinais possuem assinaturas temporais únicas (formas) que os distinguem do ruído aleatório.

### 🔬 Análise de Ruído Comparativa: PCA e Espectro (5 Ruídos)

O resultado da Projeção PCA e FFT é fisicamente consistente, refletindo a diferença real na magnitude da variação entre os comprimentos de onda UV Extremo ($\approx 171 \text{ \AA}$) e UV Próximo ($\approx 2310 \text{ \AA}$).

#### 1. Contexto Físico dos Dados

| Ruído | Comprimento de Onda | Característica Física |
| :---: | :---: | :---: |
| **R1, R2, R3, R4** | $\approx 171 \text{ \AA}$ (UV Extremo) | Baixa emissão da coroa; ruído de base baixo. |
| **R5** | $\approx 2310 \text{ \AA}$ (UV Próximo) | Alta emissão da fotosfera/cromosfera; ruído de rotação estelar alto. |

#### 2. Interpretação da Projeção PCA (PC1 vs PC2)

A PCA está correta ao refletir a diferença de amplitude bruta (variação) entre os canais:

* **Componente Principal 1 (PC1: $\mathbf{82.0\%}$ da Variância):** O PC1 é dominado pela **diferença na amplitude do ruído entre os comprimentos de onda**. O Ruído 5 (UV Próximo) possui uma variação inerente (sinal de rotação estelar e ruído instrumental) muito maior do que qualquer outro sinal. Isso o isola no extremo do eixo PC1.
    * **Conclusão:** O PC1 distingue o **canal de observação** ($\approx 2310 \text{ \AA}$ vs. $\approx 171 \text{ \AA}$), não a anomalia em si.
* **Componente Principal 2 (PC2: $\mathbf{9.2\%}$ da Variância):** O PC2, que carrega a segunda maior informação, reflete as variações de **forma e picos** dentro do cluster de baixa amplitude.
    * **Ruído 4 (Outlier PC2):** Posicionado no topo, indicando que este sinal tem uma **assinatura temporal ou geométrica única** que o separa do ruído aleatório (R1, R2, R3).

#### 3. Análise do Espectro de Frequência (FFT)

* **Ruído 5 (Magneta):** O pico de alta magnitude nas baixas frequências (próximo de $0.0$) é a **assinatura da modulação de rotação estelar** remanescente no sinal de UV Próximo.
* **Ruídos 1, 2, 3, 4:** A baixa magnitude espectral desses ruídos confirma que a variação observada no UV Extremo ($\approx 171 \text{ \AA}$) é pequena e mais próxima do ruído branco/aleatório, comparado ao Ruído 5.

#### 4. Conclusão Final

O resultado do PCA é **robusto** e reflete a realidade física: o sinal de $\approx 2310 \text{ \AA}$ é estatisticamente distinto e dominante. O foco da análise agora deve ser no PC2 para diferenciar as **formas** dos *dimmings* no canal de UV Extremo, e na análise do Ruído 5 para quantificar a **amplitude** do sinal de rotação.